# Trustworthy CXR VLM — S0–S3 generation and scoring on Kaggle (FP16)

Runs the final evaluation end to end (DR-001): FP16 generation for S0 and S1 (+5 samples for S3), then
CheXbert / BLEU / ROUGE-L / RadGraph scoring, the val-fitted review flag, Grad-CAM localization on test,
the S0–S3 ablation table with MTS, and paired bootstrap CIs. Everything lands in one zip.

**Before running**
1. Settings → Accelerator: **GPU T4 x2** (LLaVA-Med 7B in FP16 does not fit one T4). Internet: **On**.
2. Add the IU X-Ray dataset as an input (the one with `indiana_reports.csv`, `indiana_projections.csv`
   and `images/images_normalized/`) (its path is found automatically).
3. Upload the local `checkpoints/xai_head.pt` (the Grad-CAM head trained on the train split, git-ignored)
   as a small Kaggle dataset, add it as an input (found automatically). The head is never retrained here.
4. The prompt used is `vlm.prompt_template_id` in `configs/experiment_config.yaml` of the cloned commit.

The full run needs about 17 GPU hours, more than one 12-hour Kaggle session. Generation therefore stops
cleanly at `GEN_BUDGET_H` hours, so every session ends normally and keeps its output. To continue, attach the
previous version's output as an input (Add Input → Your Work → this notebook) and Save & Run All again: finished
patients are found and skipped automatically. Scoring runs only in the session where generation completes;
`phase2_results.zip` is the file to download from that last session.

In [ ]:
import glob, os, time
START = time.time()
GEN_BUDGET_H = 10.0          # stop starting new patients after this many hours; the rest is left for the next session
REPO_URL = "https://github.com/justinthomas11/mts-medical-vlm.git"
REPO_COMMIT = None            # set to the exact commit hash you evaluated locally, for reproducibility
# Inputs are found wherever Kaggle mounts them (/kaggle/input/<slug>/ or /kaggle/input/datasets/<user>/<slug>/).
IU_DATASET_DIR = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
HEAD_PATH = glob.glob("/kaggle/input/**/xai_head.pt", recursive=True)[0]   # uploaded checkpoints/xai_head.pt (train split only)
print("IU_DATASET_DIR =", IU_DATASET_DIR)
print("HEAD_PATH      =", HEAD_PATH)
RESUME_DIR = None             # None = find previous generations in any attached input automatically
QUANT = "fp16"
WORK = "/kaggle/working/mts-medical-vlm"

In [ ]:
import os, subprocess, sys
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
if REPO_COMMIT:
    subprocess.run(["git", "-C", WORK, "checkout", REPO_COMMIT], check=True)
os.chdir(WORK)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Same transformers version as local development (DR-022); fp16 needs no bitsandbytes.
# Scoring dependencies are installed up front so a broken install fails now, not after generation.
!pip install -q "transformers==5.17.0" accelerate "torchxrayvision==1.5.4" "pycocoevalcap==1.2"
!pip install -q --no-deps "f1chexbert==0.0.2" "appdirs==1.4.4"
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.device_count(), "GPU(s)",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

# RadGraph's bundled AllenNLP code needs transformers < 5, so it gets its own venv (DR-022).
RG_VENV = "/tmp/radgraph-venv"
RG_PY = os.path.join(RG_VENV, "bin", "python")
if not os.path.exists(RG_PY):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", "-q", RG_VENV], check=True)
    subprocess.run([RG_PY, "-m", "pip", "install", "-q", "torch==2.14.0",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run([RG_PY, "-m", "pip", "install", "-q", "-r", "requirements-radgraph.txt"], check=True)
subprocess.run([RG_PY, "-c", "import radgraph, transformers; print('radgraph venv ok, transformers', transformers.__version__)"],
               check=True)
os.environ["RADGRAPH_PYTHON"] = RG_PY
assert os.path.exists(HEAD_PATH), "Add the uploaded xai_head.pt as an input and set HEAD_PATH"

In [ ]:
# Link the raw IU X-Ray files where configs/data_config.yaml expects them (data/raw is git-ignored).
os.makedirs("data/raw/images", exist_ok=True)
for src, dst in [("indiana_reports.csv", "data/raw/indiana_reports.csv"),
                 ("indiana_projections.csv", "data/raw/indiana_projections.csv"),
                 ("images/images_normalized", "data/raw/images/images_normalized")]:
    if not os.path.exists(dst):
        os.symlink(os.path.join(IU_DATASET_DIR, src), dst)
print(len(os.listdir("data/raw/images/images_normalized")), "images linked (expected 7,470)")

In [ ]:
# Rebuild data/processed with the fixed seed, then STOP unless the split is identical to the committed one.
subprocess.run([sys.executable, "src/data/split_dataset.py"], check=True)
check = subprocess.run([sys.executable, "src/pipeline/verify_splits.py"])
assert check.returncode == 0, "Split differs from the committed benchmark — do not run any stage."

In [ ]:
# Full test suite on Kaggle (torch is blocked on the dev laptop). Informational: does not stop the run.
tests = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "-p", "no:cacheprovider"],
                       capture_output=True, text=True)
print(tests.stdout[-3000:])

In [ ]:
# Resume: copy generations from previous sessions so finished patients are skipped.
import glob, shutil
pattern = os.path.join(RESUME_DIR or "/kaggle/input", "**", "results", "s[01]", "generations_*.jsonl")
best = {}
for f in glob.glob(pattern, recursive=True):
    name = os.path.join(os.path.basename(os.path.dirname(f)), os.path.basename(f))   # e.g. s1/generations_test.jsonl
    if os.path.basename(f) not in ("generations_test.jsonl", "generations_val.jsonl"):
        continue
    n = sum(1 for line in open(f, encoding="utf-8") if line.strip())
    if n > best.get(name, ("", -1))[1]:
        best[name] = (f, n)
for name, (f, n) in sorted(best.items()):
    dst = os.path.join("results", name)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(f, dst)
    print(f"resumed {dst}: {n} patients (from {f})")
if not best:
    print("No previous generations attached: starting from scratch.")

In [ ]:
DEADLINE = START + GEN_BUDGET_H * 3600

def run(*args):
    cmd = [sys.executable, "src/pipeline/run_generation.py", *args, "--quant", QUANT, "--deadline", str(DEADLINE)]
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

run("--stage", "s0", "--split", "test")                    # S0: greedy, 734 test patients
run("--stage", "s1", "--split", "val", "--samples", "5")   # S1+samples on val: fits the S3 review flag (DR-020)
run("--stage", "s1", "--split", "test", "--samples", "5")  # S1 test; S2 reuses it, S3 adds the samples

def n_done(path):
    return sum(1 for line in open(path, encoding="utf-8") if line.strip()) if os.path.exists(path) else 0

EXPECTED = {"results/s0/generations_test.jsonl": 734, "results/s1/generations_val.jsonl": 366,
            "results/s1/generations_test.jsonl": 734}
progress = {f: n_done(f) for f in EXPECTED}
COMPLETE = all(progress[f] == n for f, n in EXPECTED.items())
for f, n in EXPECTED.items():
    print(f"{f}: {progress[f]}/{n}")
print("GENERATION COMPLETE: scoring follows." if COMPLETE else
      "GENERATION NOT FINISHED: scoring skipped. Attach this version's output as an input and Save & Run All again.")

## Scoring

CheXbert labels both reference and generated reports (DR-013, DR-017). The S3 review flag is fitted on
val only and applied unchanged to test (DR-020).

In [ ]:
import json, glob, shutil

In [ ]:
if not COMPLETE:
    print("Skipped: generation not finished in this session.")
else:
    def py(script, *args):
        cmd = [sys.executable, script, *args]
        print(" ".join(cmd), flush=True)
        subprocess.run(cmd, check=True)

    py("src/evaluation/evaluate_stage.py", "results/s0/generations_test.jsonl")
    py("src/evaluation/evaluate_stage.py", "results/s1/generations_test.jsonl")
    py("src/evaluation/evaluate_stage.py", "results/s1/generations_val.jsonl", "--uncertainty", "--fit-flagger")
    py("src/evaluation/evaluate_stage.py", "results/s1/generations_test.jsonl", "--uncertainty")

In [ ]:
if not COMPLETE:
    print("Skipped: generation not finished in this session.")
else:
    # S2 localization. Features go to /tmp (about 4 GB) so they are not part of the notebook output.
    os.makedirs("checkpoints", exist_ok=True)
    shutil.copy(HEAD_PATH, "checkpoints/xai_head.pt")
    os.makedirs("/tmp/features", exist_ok=True)
    if not os.path.exists("results/features"):
        os.symlink("/tmp/features", "results/features")
    py("src/pipeline/extract_features.py", "--only", "clip")

    # Sanity check: recompute val localization with Kaggle-extracted features and compare with the committed run.
    py("src/pipeline/eval_localization.py", "--split", "val", "--tag", "kaggle_check")
    for name in ["localization_val_summary.json", "localization_val_kaggle_check_summary.json"]:
        o = json.load(open(os.path.join("results/s2", name)))["all_positive_pairs"]["overall"]
        print(f"{name}: n={o['n']} pointing_game={o['pointing_game']:.4f} smr={o['smr']:.4f}")

    py("src/pipeline/eval_localization.py", "--split", "test")

In [ ]:
if not COMPLETE:
    print("Skipped: generation not finished in this session.")
else:
    py("src/evaluation/build_ablation.py", "--split", "test")
    py("src/evaluation/bootstrap_ci.py", "--split", "test")
    print(open("results/ablation/ablation_test.md", encoding="utf-8").read())

In [ ]:
# Package generations, run configs and every scored output (no features, no checkpoints) for download.
import zipfile
out = "/kaggle/working/phase2_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for d in ["s0", "s1", "s2", "s3", "ablation"]:
        for f in sorted(glob.glob(f"results/{d}/**/*", recursive=True)):
            if os.path.isfile(f):
                z.write(f)
                print("added", f)
    z.writestr("results/KAGGLE_RUN.txt",
               subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
print(out)

## After the last session

Download `phase2_results.zip` from the version whose log says **GENERATION COMPLETE** (earlier sessions only
hold partial generations). Unzip it into the local repo root (it restores `results/s0`, `s1`, `s2`, `s3` and
`ablation`), check `git status`, run `python -m pytest tests/ -v`, and commit the results.
`results/ablation/ablation_test.md` is the final S0–S3 table; `bootstrap_test.json` holds the paired CIs.